# 從 jobs_rows 擷取 tools 技能 → 彙整為 skill 清單

- 讀取 `jobs_rows.csv` 的 **tools** 欄位
- 拆成單一技能、彙整成 DataFrame（技能名稱 + 出現次數）
- 與既有 `skill_master_rows.csv`（40 項）對照，方便決定是否寫入 `skill_master` 表
- 工作目錄：在 `supabase_control` 下執行

## 1. 載入套件與路徑

In [1]:
import pandas as pd
import os

# 在 supabase_control 目錄下執行此 notebook 時，當前目錄即為 DATA_DIR
DATA_DIR = os.getcwd()
JOBS_CSV = os.path.join(DATA_DIR, "jobs_rows.csv")
SKILL_MASTER_CSV = os.path.join(DATA_DIR, "skill_master_rows.csv")

print("JOBS_CSV:", JOBS_CSV)
print("SKILL_MASTER_CSV:", SKILL_MASTER_CSV)

JOBS_CSV: c:\Users\student\final\supabase_control\jobs_rows.csv
SKILL_MASTER_CSV: c:\Users\student\final\supabase_control\skill_master_rows.csv


## 2. 讀取 jobs_rows.csv，只取 tools 欄位

In [8]:
# 只讀取必要欄位以節省記憶體
jobs = pd.read_csv(JOBS_CSV, usecols=["job_id","job_name", "tools","url"], encoding="utf-8", low_memory=False)
print(f"✓ 載入 {len(jobs):,} 筆 jobs")
print("tools 非空筆數:", jobs["tools"].notna().sum())
jobs.head(10)

✓ 載入 18,729 筆 jobs
tools 非空筆數: 12657


,job_id,job_name,url,tools
0,109g12026-01-21,C#軟體工程師,https://www.104.com.tw/job/109g1?jobsource=job...,Windows 2003、Windows XP、ASP.NET、C#、Visual Basi...
1,10ccd2026-02-02,[NB]軟體品保工程師(士林),https://www.104.com.tw/job/10ccd?jobsource=job...,NaN
2,1vmnb2026-01-30,測試人員-實習/工讀 (中興新村南投高等研究園區),https://www.104.com.tw/job/1vmnb?jobsource=job...,NaN
3,1wlvg2026-01-30,網管與雲端專員,https://www.104.com.tw/job/1wlvg?jobsource=job...,Microsoft Azure、AWS、Security、Excel、PowerPoint、...
4,1x1xw2026-02-03,專案管理師｜AI技能 + 專案歷練一次到位,https://www.104.com.tw/job/1x1xw?jobsource=job...,NaN
5,1x89f2026-01-26,研發替代役-軟體工程師,https://www.104.com.tw/job/1x89f?jobsource=job...,NaN
6,1x89f2026-02-02,研發替代役-軟體工程師,https://www.104.com.tw/job/1x89f?jobsource=job...,NaN
7,1xpg22026-01-16,系統工程師(高雄),https://www.104.com.tw/job/1xpg2?jobsource=job...,Windows Server 2019、C++、Visual Basic、MS SQL、PLC
8,1xx4g2026-02-02,繪圖人員,https://www.104.com.tw/job/1xx4g?jobsource=job...,Excel、Outlook、Word、Adobe Photoshop、AutoCAD
9,1yf7o2025-12-10,資料庫程式工程師,https://www.104.com.tw/job/1yf7o?jobsource=job...,DB2、MS SQL、Oracle


## 3. 擷取 tools：依「、」與「,」拆成單一技能，彙整成 df

In [9]:
def split_tools(s):
    """將一筆 tools 字串拆成 list（支援全形頓號、半形逗號）。"""
    if pd.isna(s) or not str(s).strip():
        return []
    s = str(s).strip()
    # 先統一用逗號切
    for sep in ["、", ","]:
        s = s.replace(sep, "|")
    return [x.strip() for x in s.split("|") if x.strip()]

# 每筆 jobs 的 tools 拆成 list
all_tools = []
for v in jobs["tools"].dropna():
    all_tools.extend(split_tools(v))

print(f"拆出之技能總數（含重複）: {len(all_tools):,}")
print(f"唯一技能數: {len(set(all_tools)):,}")

拆出之技能總數（含重複）: 61,124
唯一技能數: 512


In [10]:
# 彙整：技能名稱 + 出現次數，並加索引（從 1 開始）
from collections import Counter

counts = Counter(all_tools)
df_tools = (
    pd.DataFrame([{"skill_name": k, "count": v} for k, v in counts.items()])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
df_tools.index = df_tools.index + 1
df_tools.index.name = "index"

print("技能 DataFrame（前 50 筆）:")
display(df_tools.head(50))
print(f"\n共 {len(df_tools):,} 個不重複技能")
df_tools

技能 DataFrame（前 50 筆）:


,skill_name,count
index,,
1,Python,3316
2,JavaScript,3045
3,C#,2641
4,Git,2321
5,C++,2320
6,HTML,2066
7,MS SQL,2045
8,Excel,2014
9,Linux,1950



共 512 個不重複技能


,skill_name,count
index,,
1,Python,3316
2,JavaScript,3045
3,C#,2641
4,Git,2321
5,C++,2320
...,...,...
508,Visual J++,1
509,Pascal,1
510,InVision,1


## 4. 與既有 skill_master（40 項）對照

In [11]:
skill_master = pd.read_csv(SKILL_MASTER_CSV, encoding="utf-8")
existing_skills = set(skill_master["skill_name"].str.strip().str.lower())
print(f"既有的 skill_master 共 {len(skill_master)} 項")
skill_master.head()

既有的 skill_master 共 40 項


,skill_id,skill_name,skill_category,synonyms,created_at
0,1,Python,Programming,"[""python"",""py""]",2026-01-29 02:26:28.199447+00
1,2,JavaScript,Programming,"[""JS"",""js"",""javascript"",""ECMAScript""]",2026-01-29 02:26:28.199447+00
2,3,Java,Programming,"[""java""]",2026-01-29 02:26:28.199447+00
3,4,TypeScript,Programming,"[""TS"",""ts"",""typescript""]",2026-01-29 02:26:28.199447+00
4,5,C++,Programming,"[""cpp"",""c plus plus""]",2026-01-29 02:26:28.199447+00


In [12]:
# 標記：是否已在 skill_master 中（以名稱不分大小寫比對）
df_tools["in_skill_master"] = df_tools["skill_name"].str.strip().str.lower().isin(existing_skills)

in_master = df_tools["in_skill_master"].sum()
not_in_master = (~df_tools["in_skill_master"]).sum()
print(f"在 jobs 的 tools 中：已在 skill_master 的 {in_master} 個，尚未在 skill_master 的 {not_in_master} 個")

df_tools.head(80)

在 jobs 的 tools 中：已在 skill_master 的 18 個，尚未在 skill_master 的 494 個


,skill_name,count,in_skill_master
index,,,
1,Python,3316,True
2,JavaScript,3045,True
3,C#,2641,False
4,Git,2321,True
5,C++,2320,True
...,...,...,...
76,Objective-C,121,False
77,JIRA,115,False
78,Vmware,114,False


In [13]:
# 僅列出「尚未在 skill_master」的技能（可作為待寫入候選）
df_new_only = df_tools[~df_tools["in_skill_master"]].copy()
print("尚未在 skill_master 的技能（前 100 筆，依出現次數排序）:")
display(df_new_only.head(100))
print(f"\n共 {len(df_new_only):,} 個候選可考慮寫入 skill_master")
df_new_only

尚未在 skill_master 的技能（前 100 筆，依出現次數排序）:


,skill_name,count,in_skill_master
index,,,
3,C#,2641,False
6,HTML,2066,False
7,MS SQL,2045,False
8,Excel,2014,False
9,Linux,1950,False
...,...,...,...
114,scikit-learn,67,False
115,Project,66,False
116,SAS,66,False



共 494 個候選可考慮寫入 skill_master


,skill_name,count,in_skill_master
index,,,
3,C#,2641,False
6,HTML,2066,False
7,MS SQL,2045,False
8,Excel,2014,False
9,Linux,1950,False
...,...,...,...
508,Visual J++,1,False
509,Pascal,1,False
510,InVision,1,False


## 5. 匯出（可選）
若要先把彙整結果存成 CSV 再決定寫入哪些到 DB，可執行下面格子。

In [14]:
# 匯出完整 df_tools（含 in_skill_master 標記）
out_path = os.path.join(DATA_DIR, "tools_extracted_skills.csv")
df_tools.to_csv(out_path, encoding="utf-8-sig")
print(f"已匯出: {out_path}")

# 僅匯出「尚未在 skill_master」的候選
out_new = os.path.join(DATA_DIR, "tools_new_candidates.csv")
df_new_only.to_csv(out_new, encoding="utf-8-sig")
print(f"已匯出候選清單: {out_new}")

已匯出: c:\Users\student\final\supabase_control\tools_extracted_skills.csv
已匯出候選清單: c:\Users\student\final\supabase_control\tools_new_candidates.csv


## 6. 寫入 DB 評估（職缺推薦 / 課程推薦）

以下載入「建議寫入清單」：`skill_master_recommended_to_write.csv`  
- **WRITE_NEW**：建議新增一筆到 `skill_master`（含 Linux、C#、HTML、CSS、Shell、PyTorch 等）  
- **ADD_SYNONYM_ONLY**：職缺常寫的名稱，只須在既有 skill 的 `synonyms` 加入，不新增一筆  
詳細準則與排除原因見 `skill_write_evaluation.md`。

In [15]:
rec_path = os.path.join(DATA_DIR, "skill_master_recommended_to_write.csv")
df_rec = pd.read_csv(rec_path, encoding="utf-8-sig")
print("建議寫入清單摘要：")
print(df_rec["action"].value_counts())
print("\n--- WRITE_NEW（建議新增至 skill_master）前 25 筆 ---")
display(df_rec[df_rec["action"] == "WRITE_NEW"][["skill_name", "count", "suggested_category", "note"]].head(25))
print("\n--- ADD_SYNONYM_ONLY（只更新既有 skill 的 synonyms）---")
display(df_rec[df_rec["action"] == "ADD_SYNONYM_ONLY"][["skill_name", "count", "note"]])

建議寫入清單摘要：
action
WRITE_NEW           53
ADD_SYNONYM_ONLY     6
Name: count, dtype: int64

--- WRITE_NEW（建議新增至 skill_master）前 25 筆 ---


,skill_name,count,suggested_category,note
0,Linux,1950,Tool,職缺多、與 DevOps/後端/嵌入式相關，強烈建議
1,C#,2641,Programming,高職缺數
2,HTML,2066,Programming,前端必備
3,CSS,1796,Programming,前端必備
5,Excel,2014,Tool,辦公室/數據分析
6,C,1633,Programming,系統/嵌入式
7,PowerPoint,1508,Tool,辦公室
8,Word,1715,Tool,辦公室
10,jQuery,1246,Framework,前端
12,Outlook,680,Tool,辦公室



--- ADD_SYNONYM_ONLY（只更新既有 skill 的 synonyms）---


,skill_name,count,note
4,MS SQL,2045,對應既有 SQL Server，在 synonyms 加 MS SQL/MSSQL
9,Github,822,對應既有 Git，在 synonyms 加 Github/GitHub
11,VueJS,914,對應既有 Vue，已有 VueJS 可確認
13,ReactJS,647,對應既有 React，已有 ReactJS 可確認
40,AngularJS,165,對應既有 Angular
41,Microsoft Azure,107,對應既有 Azure


## 7. 寫入 DB（依 skill_write_evaluation.md 步驟）

1. **Step 1**：將 `WRITE_NEW` 的項目**新增**至 `skill_master`（skill_name, skill_category, synonyms）。
2. **Step 2**：將 `ADD_SYNONYM_ONLY` 的職缺常見名稱**併入既有技能**的 `synonyms`，不新增一筆。

In [ ]:
# 載入建議寫入清單、連線 Supabase（需先執行上方 cells 取得 DATA_DIR）
import json
import sys

if DATA_DIR not in sys.path:
    sys.path.insert(0, DATA_DIR)
from supabase_connection import connect_to_supabase

rec_path = os.path.join(DATA_DIR, "skill_master_recommended_to_write.csv")
if os.path.exists(rec_path):
    df_rec = pd.read_csv(rec_path, encoding="utf-8-sig")
    print(f"✓ 已從 CSV 載入建議清單: {len(df_rec)} 筆")
else:
    # 若 CSV 不存在，依 skill_write_evaluation.md 建立內建清單
    df_rec = pd.DataFrame([
        {"skill_name": "Linux", "count": 1950, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["linux","LINUX"]', "note": "強烈建議"},
        {"skill_name": "C#", "count": 2641, "action": "WRITE_NEW", "suggested_category": "Programming", "synonyms": '["c#","C Sharp"]', "note": ""},
        {"skill_name": "HTML", "count": 2066, "action": "WRITE_NEW", "suggested_category": "Programming", "synonyms": '["html","HTML5"]', "note": ""},
        {"skill_name": "CSS", "count": 1796, "action": "WRITE_NEW", "suggested_category": "Programming", "synonyms": '["css","CSS3"]', "note": ""},
        {"skill_name": "MS SQL", "count": 2045, "action": "ADD_SYNONYM_ONLY", "suggested_category": "", "synonyms": "", "note": "對應 SQL Server"},
        {"skill_name": "Excel", "count": 2014, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["excel","Microsoft Excel"]', "note": ""},
        {"skill_name": "C", "count": 1633, "action": "WRITE_NEW", "suggested_category": "Programming", "synonyms": '["c","C language"]', "note": ""},
        {"skill_name": "PowerPoint", "count": 1508, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["powerpoint","PPT"]', "note": ""},
        {"skill_name": "Word", "count": 1715, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["word","Microsoft Word"]', "note": ""},
        {"skill_name": "Github", "count": 822, "action": "ADD_SYNONYM_ONLY", "suggested_category": "", "synonyms": "", "note": "對應 Git"},
        {"skill_name": "jQuery", "count": 1246, "action": "WRITE_NEW", "suggested_category": "Framework", "synonyms": '["jquery","jQuery"]', "note": ""},
        {"skill_name": "VueJS", "count": 914, "action": "ADD_SYNONYM_ONLY", "suggested_category": "", "synonyms": "", "note": "對應 Vue"},
        {"skill_name": "Outlook", "count": 680, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["outlook","Microsoft Outlook"]', "note": ""},
        {"skill_name": "ReactJS", "count": 647, "action": "ADD_SYNONYM_ONLY", "suggested_category": "", "synonyms": "", "note": "對應 React"},
        {"skill_name": "Shell", "count": 270, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["shell","bash","Shell Script"]', "note": ""},
        {"skill_name": "PyTorch", "count": 277, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["pytorch","PyTorch"]', "note": ""},
        {"skill_name": "TensorFlow", "count": 217, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["tensorflow","tensor flow"]', "note": ""},
        {"skill_name": "Power BI", "count": 262, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["power bi","PowerBI"]', "note": ""},
        {"skill_name": "Tableau", "count": 214, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["tableau"]', "note": ""},
        {"skill_name": "ETL", "count": 210, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["etl"]', "note": ""},
        {"skill_name": "Figma", "count": 201, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["figma"]', "note": ""},
        {"skill_name": "LLM", "count": 348, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["llm","Large Language Model"]', "note": ""},
        {"skill_name": "Android", "count": 446, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["android","Android SDK"]', "note": ""},
        {"skill_name": "iOS", "count": 309, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["ios","iOS"]', "note": ""},
        {"skill_name": "JIRA", "count": 115, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["jira","Jira"]', "note": ""},
        {"skill_name": "Vmware", "count": 114, "action": "WRITE_NEW", "suggested_category": "Tool", "synonyms": '["vmware","VMware","ESXi"]', "note": ""},
        {"skill_name": "Microsoft Azure", "count": 107, "action": "ADD_SYNONYM_ONLY", "suggested_category": "", "synonyms": "", "note": "對應 Azure"},
        {"skill_name": "AngularJS", "count": 165, "action": "ADD_SYNONYM_ONLY", "suggested_category": "", "synonyms": "", "note": "對應 Angular"},
    ])
    print("✓ 使用內建建議清單（CSV 不存在）")

supabase = connect_to_supabase()
df_write = df_rec[df_rec["action"] == "WRITE_NEW"].copy()
df_syn = df_rec[df_rec["action"] == "ADD_SYNONYM_ONLY"].copy()
print(f"WRITE_NEW: {len(df_write)} 筆, ADD_SYNONYM_ONLY: {len(df_syn)} 筆")

### Step 1：新增 WRITE_NEW 技能至 skill_master

依建議清單逐筆 insert（skill_name, skill_category, synonyms）。若該 skill_name 已存在則跳過，避免重複。

In [ ]:
def parse_synonyms(s):
    """將 CSV 的 synonyms 欄位轉成 list（支援 JSON 字串）。"""
    if pd.isna(s) or not str(s).strip():
        return []
    s = str(s).strip()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        try:
            import ast
            return ast.literal_eval(s)
        except Exception:
            return []

# 取得目前 skill_master 已有之名稱（小寫），用於跳過已存在
existing = supabase.table("skill_master").select("skill_name").execute()
existing_names_lower = {r["skill_name"].strip().lower() for r in existing.data}

inserted = 0
skipped = 0
for _, row in df_write.iterrows():
    name = str(row["skill_name"]).strip()
    if name.lower() in existing_names_lower:
        skipped += 1
        continue
    cat = str(row["suggested_category"]).strip() if pd.notna(row["suggested_category"]) else "Tool"
    syn = parse_synonyms(row.get("synonyms", ""))
    payload = {"skill_name": name, "skill_category": cat, "synonyms": syn}
    try:
        supabase.table("skill_master").insert(payload).execute()
        existing_names_lower.add(name.lower())
        inserted += 1
        print(f"  新增: {name} ({cat})")
    except Exception as e:
        print(f"  跳過/錯誤 {name}: {e}")
        skipped += 1

print(f"\n✓ Step 1 完成 — 新增 {inserted} 筆，跳過/錯誤 {skipped} 筆")

### Step 2：更新既有技能的 synonyms（ADD_SYNONYM_ONLY）

將職缺常見名稱（如 MS SQL、Github、ReactJS）加入對應既有 skill 的 synonyms，不新增一筆。

In [ ]:
# 職缺常見名稱 -> 對應既有 skill_master 的 skill_name（依 skill_write_evaluation.md）
JOB_NAME_TO_MASTER = {
    "MS SQL": "SQL Server",
    "Github": "Git",
    "ReactJS": "React",
    "VueJS": "Vue",
    "Microsoft Azure": "Azure",
    "AngularJS": "Angular",
}

# 要併入的 synonym 字串（職缺名稱 + 常見變體）
SYNONYM_ADDITIONS = {
    "MS SQL": ["MS SQL", "MSSQL", "mssql"],
    "Github": ["Github", "GitHub", "github"],
    "ReactJS": ["ReactJS", "reactjs"],
    "VueJS": ["VueJS", "vuejs"],
    "Microsoft Azure": ["Microsoft Azure", "microsoft azure"],
    "AngularJS": ["AngularJS", "angularjs"],
}

updated = 0
for job_name, master_name in JOB_NAME_TO_MASTER.items():
    res = supabase.table("skill_master").select("skill_id, skill_name, synonyms").eq("skill_name", master_name).execute()
    if not res.data or len(res.data) == 0:
        print(f"  找不到既有技能: {master_name}，跳過 {job_name}")
        continue
    row = res.data[0]
    skill_id = row["skill_id"]
    current = row.get("synonyms") or []
    if not isinstance(current, list):
        current = list(current) if current else []
    to_add = SYNONYM_ADDITIONS.get(job_name, [job_name])
    new_synonyms = list(current)
    for s in to_add:
        if s and s not in new_synonyms:
            new_synonyms.append(s)
    if new_synonyms == current:
        print(f"  {master_name}: synonyms 無變更")
        continue
    try:
        supabase.table("skill_master").update({"synonyms": new_synonyms}).eq("skill_id", skill_id).execute()
        updated += 1
        print(f"  已更新 {master_name} (skill_id={skill_id})，synonyms +{len(new_synonyms) - len(current)} 個")
    except Exception as e:
        print(f"  更新失敗 {master_name}: {e}")

print(f"\n✓ Step 2 完成 — 已更新 {updated} 筆既有技能的 synonyms")

### 寫入後檢查

查詢 `skill_master` 筆數與部分新寫入項目（例如 Linux、C#）。

In [ ]:
all_skills = supabase.table("skill_master").select("skill_id, skill_name, skill_category").order("skill_id").execute()
print(f"skill_master 目前共 {len(all_skills.data)} 筆")
# 顯示名稱含 Linux / C# / Shell 的項目
for r in all_skills.data:
    n = r.get("skill_name") or ""
    if "Linux" in n or "C#" in n or "Shell" in n or "PyTorch" in n:
        print(f"  {r['skill_id']}: {r['skill_name']} ({r.get('skill_category','')})")